# MLOps: Why Production ML Is Different

Production ML is harder than academic ML not because of algorithms but because of the systems around them: data pipelines, training infrastructure, versioning, monitoring, and the unique failure modes that emerge when ML models interact with live data and user behavior.

## What Interviewers Test
- Understanding of the ML lifecycle and where it breaks in production
- Training-serving skew: what it is and how to prevent it
- Technical debt patterns unique to ML (data dependencies, entanglement)
- MLOps maturity model: manual → pipeline automation → full CI/CD
- Tool landscape by category (not vendor names — categories)

## The ML Lifecycle

```
Problem Definition → Data Collection → Feature Engineering
       ↓
Model Training → Evaluation → Deployment
       ↓
Monitoring → Retraining (back to Data Collection)
```

**Where it breaks in production:**
- Data schema changes silently (upstream system update)
- Feature computation differs between training and serving (skew)
- Model degrades as real-world distribution shifts
- Ground truth labels arrive late or not at all
- Dependencies between features create hidden coupling (entanglement)


## Training-Serving Skew Taxonomy

| Type | Cause | Detection | Prevention |
|---|---|---|---|
| **Data skew** | Different preprocessing code in training vs serving | Feature distribution monitoring | Shared feature computation code |
| **Schema skew** | Feature schema changed in serving but not retraining | Schema validation | Versioned schemas |
| **Distribution shift** | Real world changed; model trained on stale data | PSI/KS drift detection | Retraining triggers |
| **Label skew** | Label definition changed | Label distribution monitoring | Label versioning |
| **Online skew** | Features unavailable at serving time (future leakage in training) | Point-in-time eval | Feature store with PIT correctness |

> 💡 **Interview Tip:** Training-serving skew is the #1 production ML failure mode. When asked "what goes wrong in production?", skew is always part of the answer.


In [ ]:
# Illustrative: detect skew between training and serving feature distributions
import numpy as np
from scipy import stats

np.random.seed(42)

def population_stability_index(expected, actual, n_bins=10):
    bins = np.linspace(min(expected.min(), actual.min()),
                       max(expected.max(), actual.max()) + 1e-6, n_bins+1)
    exp_pct = np.histogram(expected, bins)[0] / len(expected)
    act_pct = np.histogram(actual,   bins)[0] / len(actual)
    exp_pct = np.clip(exp_pct, 1e-6, None)
    act_pct = np.clip(act_pct, 1e-6, None)
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))

# Training distribution (feature_x at training time)
train_feat = np.random.normal(loc=0.0, scale=1.0, size=5000)
# Serving distribution (small shift - acceptable)
serve_feat_ok = np.random.normal(loc=0.1, scale=1.0, size=2000)
# Serving distribution (large shift - retrain needed)
serve_feat_bad = np.random.normal(loc=2.0, scale=1.5, size=2000)

psi_ok  = population_stability_index(train_feat, serve_feat_ok)
psi_bad = population_stability_index(train_feat, serve_feat_bad)

print(f"PSI (stable):   {psi_ok:.4f}  → {'OK' if psi_ok < 0.1 else 'INVESTIGATE'}")
print(f"PSI (drifted):  {psi_bad:.4f}  → {'RETRAIN' if psi_bad > 0.2 else 'INVESTIGATE'}")
print()
print("Thresholds: PSI < 0.1 stable | 0.1-0.2 moderate shift | > 0.2 retrain")


## ML Technical Debt (Hidden Costs)

Summarizing key patterns from "Machine Learning: The High-Interest Credit Card of Technical Debt":

**1. Data dependencies (unstable):** When upstream data changes, all downstream models break silently. Test: what happens if an upstream feature column is renamed?

**2. Entanglement:** Changing any input feature changes all output predictions — features are deeply coupled. Test: can you remove one feature without affecting others?

**3. Feedback loops:** Model output influences future input data (recommendation → click → training data). Degenerate loops cause popularity bias spirals.

**4. Correction cascades:** Model A feeds model B; fixing A's outputs breaks B. Explicit or implicit dependencies between models.

**5. Undeclared consumers:** Other systems started using your model output without formal contracts. Changing output format breaks them.


## MLOps Maturity Levels

| Level | Description | Key capability |
|---|---|---|
| **Level 0** | Manual, notebook-based | Scripts run by hand; no automation |
| **Level 1** | Pipeline automation | Automated training pipeline; triggered on schedule |
| **Level 2** | CI/CD for ML | Automated testing, validation, deployment on code commit |
| **Level 3** | Continuous training** | Auto-retrain triggered by data/model drift |

Most production teams are at Level 1–2. Level 3 requires robust data pipelines and eval automation.


## Tool Landscape by Category

| Category | What it solves | Open-source examples |
|---|---|---|
| **Orchestration** | DAG scheduling and dependency management | Airflow, Prefect, Dagster |
| **Experiment tracking** | Log hyperparams, metrics, artifacts | MLflow, Weights & Biases |
| **Feature store** | Feature computation, storage, serving | Feast, Tecton (commercial) |
| **Model registry** | Version, stage, and metadata for models | MLflow Registry, SageMaker Registry |
| **Serving** | Model inference at scale | TorchServe, BentoML, Ray Serve |
| **Monitoring** | Data + model + system drift detection | WhyLogs, Evidently |
| **CI/CD for ML** | Testing and deployment pipelines | GitHub Actions, Kubeflow Pipelines |

> 💡 **Interview Tip:** Don't memorize vendor names — interviewers care that you know the *categories* and why each exists. "I'd use a feature store to solve offline/online consistency" is the right level of abstraction.


## Common Interview Questions

**Q: What is training-serving skew and how do you prevent it?**
When the feature computation during training differs from that during serving, the model sees a different input distribution than it was trained on, causing degraded performance. Prevention: use shared code for feature computation (same Python function called from both training and serving), test feature parity in CI, and monitor feature distributions at serving time against training distributions.

**Q: What is the difference between concept drift and data drift?**
Data drift (covariate shift): the input feature distribution P(X) changes. Concept drift: the relationship between inputs and outputs P(Y|X) changes. Both degrade model performance, but concept drift is harder to detect because you need labels to see it. Monitor data drift with PSI/KS; monitor concept drift with held-out labeled evaluation sets and business metrics.

**Q: What makes CI/CD for ML different from software CI/CD?**
Software CI/CD tests code logic; ML CI/CD must also test data quality (schema, distribution), model behavior (performance on eval sets, invariance tests), and infrastructure (training completes, serving latency). ML tests are also non-deterministic and slower, requiring statistical significance checks and careful caching.

**Q: What is the ML feedback loop problem?**
A model's predictions influence user behavior, which becomes future training data. If a recommendation model always surfaces the same items, users only interact with those items, so training data only covers those items, causing the model to recommend them even more. Mitigation: exploration budgets, counterfactual logging, IPS corrections.

## Key Takeaways
- Production ML fails at the system level, not the algorithm level — pipelines, skew, and monitoring
- Training-serving skew is the #1 failure mode: different code paths produce different features
- Technical debt: data dependencies, entanglement, feedback loops, correction cascades
- Maturity levels: manual (0) → pipeline (1) → CI/CD (2) → continuous training (3)
- Tool categories: orchestration, experiment tracking, feature store, registry, serving, monitoring
- PSI > 0.2 = significant drift → retrain; build this into your monitoring plan